# Stage 3 Clinical NLP & Baseline Systems — Independent Evaluation Walkthrough

**Project**: Personalized Precision Medicine for Oncology Treatment Optimization  
**Role**: Independent Evaluation Engineer  
**Status**: OFFICIAL & REPRODUCIBLE (Frozen Upstream Artifacts)  

---

## 1. Evaluation Configuration & Path Setup
Inspect the evaluation configuration, random seeds, and read-only upstream data pathways.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Setup paths
notebook_dir = Path.cwd()
eval_src = (notebook_dir.parent / "src").resolve()
if str(eval_src) not in sys.path:
    sys.path.insert(0, str(eval_src))

from config import (
    RAW_PARQUET_PATH, TRAIN_PARQUET_PATH, VAL_PARQUET_PATH, LOCKED_TEST_PARQUET_PATH,
    URGENCY_MODEL_PATH, HAZARD_MODEL_PATH, RANDOM_SEED, RESULTS_METRICS_DIR
)
print("Random Seed:", RANDOM_SEED)
print("Urgency Model Path:", URGENCY_MODEL_PATH)
print("Hazard Model Path:", HAZARD_MODEL_PATH)

## 2. Dataset Loading & Split Verification
Load Train, Validation, and Locked Test sets in strictly read-only mode and verify zero patient/encounter overlap.

In [ ]:
from data_loader import load_train_data, load_validation_data, load_locked_test_data
from leakage_checks import verify_split_isolation, verify_upstream_file_hashes

df_train = load_train_data()
df_val = load_validation_data()
df_test = load_locked_test_data()

print(f"Train Documents: {len(df_train)} | Patients: {df_train['patient_id'].nunique()}")
print(f"Validation Documents: {len(df_val)} | Patients: {df_val['patient_id'].nunique()}")
print(f"Locked Test Documents: {len(df_test)} | Patients: {df_test['patient_id'].nunique()}")

isolation_res = verify_split_isolation(df_train, df_val, df_test)
print("Completely Isolated:", isolation_res["is_completely_isolated"])
print("Patient Overlaps:", isolation_res["patient_overlap"])

hash_res = verify_upstream_file_hashes()
print("All Upstream Checksums Invariant:", hash_res["all_files_invariant"])

## 3. Deserializing Frozen NLP Artifacts
Load the frozen vectorizer, scaler, label encoders, and logistic regression baselines.

In [ ]:
from model_loader import load_frozen_artifacts
artifacts = load_frozen_artifacts()

print("Urgency Classes:", list(artifacts.urgency_encoder.classes_))
print("Hazard Classes:", list(artifacts.hazard_encoder.classes_))
print("TF-IDF Vocabulary Size:", len(artifacts.tfidf_vectorizer.vocabulary_))
print("Numeric Features Count:", len(artifacts.numeric_feature_cols))

## 4. Batch Predictions on Validation & Locked Test
Generate predictions, confidences, and full posterior class distributions.

In [ ]:
from prediction_runner import run_predictions

val_preds_df, val_urg_probs, val_haz_probs, X_val, struct_val = run_predictions(df_val, artifacts)
test_preds_df, test_urg_probs, test_haz_probs, X_test, struct_test = run_predictions(df_test, artifacts)

print("Validation Feature Matrix Shape:", X_val.shape)
print("Locked Test Feature Matrix Shape:", X_test.shape)
val_preds_df[["document_id", "ground_truth_urgency", "predicted_urgency", "urgency_confidence"]].head(5)

## 5. Classification Performance & Generalization Analysis
Compare official Validation and Locked Test classification metrics across Urgency and Hazard tasks.

In [ ]:
from classification_metrics import compute_classification_metrics

urg_classes = list(artifacts.urgency_encoder.classes_)
val_urg_metrics = compute_classification_metrics(
    val_preds_df["ground_truth_urgency"].values, val_preds_df["predicted_urgency"].values, urg_classes
)
test_urg_metrics = compute_classification_metrics(
    test_preds_df["ground_truth_urgency"].values, test_preds_df["predicted_urgency"].values, urg_classes
)

comp_df = pd.DataFrame([
    {"Metric": "Accuracy", "Validation": val_urg_metrics["accuracy"], "Locked Test": test_urg_metrics["accuracy"]},
    {"Metric": "Macro F1", "Validation": val_urg_metrics["macro_f1"], "Locked Test": test_urg_metrics["macro_f1"]},
    {"Metric": "Weighted F1", "Validation": val_urg_metrics["weighted_f1"], "Locked Test": test_urg_metrics["weighted_f1"]},
    {"Metric": "Critical Recall", "Validation": val_urg_metrics["critical_recall"], "Locked Test": test_urg_metrics["critical_recall"]}
])
print("=== TRIAGE URGENCY CLASSIFICATION GENERALIZATION ===")
display(comp_df)

## 6. Patient-Clustered 95% Bootstrap Confidence Intervals
Calculate rigorous 95% empirical confidence intervals via 1,000 patient-level resamples.

In [ ]:
from confidence_intervals import compute_patient_bootstrap_ci

ci_results = compute_patient_bootstrap_ci(
    df_test,
    test_preds_df["ground_truth_urgency"].values,
    test_preds_df["predicted_urgency"].values,
    n_iterations=1000,
    random_seed=42,
    critical_class="CRITICAL"
)

ci_table = []
for metric, data in ci_results["metrics"].items():
    ci_table.append({
        "Metric": metric.replace("_", " ").title(),
        "Mean": data["mean"],
        "Std Error": data["std_err"],
        "95% CI Lower": data["ci_lower"],
        "95% CI Upper": data["ci_upper"]
    })
display(pd.DataFrame(ci_table))

## 7. Clinical Entity Extraction & Negation Diagnostics
Inspect exact/relaxed span matches and negation rule consistency.

In [ ]:
from extraction_metrics import evaluate_dataset_extractions
from negation_metrics import evaluate_diagnostic_negation_suite

ner_relaxed = evaluate_dataset_extractions(df_test, match_type="relaxed")
print(f"Locked Test NER Relaxed F1: {ner_relaxed['macro_mean_f1']} (P={ner_relaxed['macro_mean_precision']}, R={ner_relaxed['macro_mean_recall']})")

neg_res = evaluate_diagnostic_negation_suite()
print(f"Negation Diagnostic Benchmark Accuracy: {neg_res['overall_accuracy']} ({neg_res['total_correct']}/{neg_res['total_test_cases']})")

## 8. Calibration & Robustness Evaluation
Evaluate probability calibration (ECE/Brier) and robustness against textual variations.

In [ ]:
from calibration import compute_calibration_metrics
from robustness import evaluate_perturbation_stability

test_true_idx = artifacts.urgency_encoder.transform(test_preds_df["ground_truth_urgency"])
calib_urg = compute_calibration_metrics(test_true_idx, test_urg_probs, task_name="urgency_test")
print(f"Urgency Brier Score: {calib_urg['brier_score']} | Expected Calibration Error: {calib_urg['expected_calibration_error']}")

robust_eval = evaluate_perturbation_stability(df_val, artifacts, sample_size=150)
for pert, res in robust_eval["perturbation_evaluations"].items():
    print(f"Perturbation: {pert:<20} | Urgency Agreement: {res['urgency_prediction_agreement']:.4f}")

## 9. Final Evaluation Summary & SLM Handoff Verdict
Summary of findings and handoff status.

In [ ]:
summary_path = RESULTS_METRICS_DIR / "evaluation_summary.json"
with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

print("=== FINAL EVALUATION SUMMARY ===")
print("Evaluation Status:", summary["metadata"]["eval_status"])
print("Validation Urgency Macro F1:", summary["validation_key_metrics"]["urgency_macro_f1"])
print("Locked Test Urgency Macro F1:", summary["locked_test_key_metrics"]["urgency_macro_f1"])
print("Locked Test Critical Recall:", summary["locked_test_key_metrics"]["urgency_critical_recall"])
print("Leakage Free:", summary["leakage"]["is_completely_isolated"])
print("Deterministic Reproducibility:", summary["reproducibility"]["is_perfectly_reproducible"])
print("\nVERDICT: ACCEPTED WITH LIMITATIONS (Ready for SLM fine-tuning baseline)")